In [1]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_curve, auc, classification_report
from xgboost import XGBClassifier

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_FILE = PROJECT_ROOT / "data/raw/synthetic_financial/synthetic_financial.csv"
if not DATA_FILE.exists():
    DATA_FILE = PROJECT_ROOT / "data/raw/synthetic_financial/cleaned_data.csv"

RANDOM_STATE = 42

In [2]:
df = pd.read_csv(DATA_FILE)
print(f"Loaded: {DATA_FILE}")
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]:,} columns")
print(f"Fraud cases: {int(df['isFraud'].sum()):,} ({df['isFraud'].mean() * 100:.4f}%)")

y = df["isFraud"].astype(np.int8)

# Drop high-cardinality account identifiers. They identify individual accounts but do not generalize well.
X = df.drop(columns=["isFraud", "nameOrig", "nameDest"], errors="ignore")

# Convert the transaction type into numeric model features.
X = pd.get_dummies(X, columns=["type"], dtype=np.int8)

# Keep the feature matrix numeric and memory efficient for XGBoost.
float_cols = X.select_dtypes(include=["float64"]).columns
int_cols = X.select_dtypes(include=["int64"]).columns
X[float_cols] = X[float_cols].astype(np.float32)
X[int_cols] = X[int_cols].astype(np.int32)
X = X.apply(pd.to_numeric, errors="coerce").fillna(0)

feature_names = X.columns.tolist()
print(f"Model feature matrix: {X.shape[0]:,} rows x {X.shape[1]:,} features")
print("Features:", feature_names)

Loaded: /Users/youseftahoon/DEPI Project/data/raw/synthetic_financial/synthetic_financial.csv
Shape: 6,362,620 rows x 11 columns
Fraud cases: 8,213 (0.1291%)


Model feature matrix: 6,362,620 rows x 12 features
Features: ['step', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'isFlaggedFraud', 'type_CASH_IN', 'type_CASH_OUT', 'type_DEBIT', 'type_PAYMENT', 'type_TRANSFER']


In [3]:
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Train shape:", X_train.shape)
print("Validation shape:", X_val.shape)
print("Validation fraud cases:", int(y_val.sum()))

Train shape: (5090096, 12)
Validation shape: (1272524, 12)
Validation fraud cases: 1643


In [4]:
print("NaN in Train:", X_train.isnull().sum().sum())
print("NaN in Validation:", X_val.isnull().sum().sum())

NaN in Train: 0
NaN in Validation: 0


In [5]:
ratio = float(y_train.value_counts()[0] / y_train.value_counts()[1])
print(f"Class imbalance ratio scale_pos_weight: {ratio:.2f}")

model = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    scale_pos_weight=ratio,
    eval_metric="aucpr",
    tree_method="hist",
    random_state=RANDOM_STATE,
)

model.fit(X_train, y_train)

Class imbalance ratio scale_pos_weight: 773.75


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='aucpr', feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=500,
              n_jobs=None, num_parallel_tree=None, ...)

In [6]:
import plotly.graph_objects as go

y_probs = model.predict_proba(X_val)[:, 1]

precision, recall, thresholds = precision_recall_curve(y_val, y_probs)
pr_auc = auc(recall, precision)

# Downsample only the displayed curve to keep the notebook light; PR-AUC is computed on the full curve.
plot_points = min(5_000, len(recall))
plot_idx = np.linspace(0, len(recall) - 1, plot_points).astype(int)
threshold_text = np.array(["Threshold: N/A"] * len(recall), dtype=object)
threshold_text[:len(thresholds)] = [f"Threshold: {round(t, 3)}" for t in thresholds]

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=recall[plot_idx],
    y=precision[plot_idx],
    mode='lines',
    name=f'XGBoost (AUC={pr_auc:.3f})',
    text=threshold_text[plot_idx],
    hovertemplate="<b>Recall: %{x:.3f}</b><br>Precision: %{y:.3f}<br>%{text}<extra></extra>"
))

fig.update_layout(
    title=f"Precision-Recall Curve (Interactive) - PR-AUC: {pr_auc:.4f}",
    xaxis_title="Recall",
    yaxis_title="Precision",
    hovermode="closest",
    template="plotly_dark"
)

fig.show()
print(f"PR-AUC: {pr_auc:.4f}")

PR-AUC: 0.9651


In [7]:
best_threshold = 0.971

y_pred_final = (y_probs >= best_threshold).astype(int)

print(f"Classification Report at threshold {best_threshold}:")
print(classification_report(y_val, y_pred_final))

Classification Report at threshold 0.971:


              precision    recall  f1-score   support

           0       1.00      1.00      1.00   1270881
           1       0.84      0.93      0.88      1643

    accuracy                           1.00   1272524
   macro avg       0.92      0.97      0.94   1272524
weighted avg       1.00      1.00      1.00   1272524



In [8]:
MODEL_DIR = PROJECT_ROOT / "artifacts/models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
model_path = MODEL_DIR / "synthetic_financial_xgboost.pkl"
features_path = MODEL_DIR / "synthetic_financial_xgboost_features.json"

joblib.dump(model, model_path)
features_path.write_text(json.dumps(feature_names, indent=2))

print("Saved model:", model_path)
print("Saved feature list:", features_path)

Saved model: /Users/youseftahoon/DEPI Project/artifacts/models/synthetic_financial_xgboost.pkl
Saved feature list: /Users/youseftahoon/DEPI Project/artifacts/models/synthetic_financial_xgboost_features.json
